# Feature Selection Methods: Comparison Study

**Objective:** Compare two feature selection methods for customer classification task
- **Method 1:** Correlation-Based (SelectKBest with f_classif)
- **Method 2:** Backward Elimination (stepwise removal)

**Task:** Identify top features for predicting Champions customers

**Rubric:** Feature Selection (15%) - EDA + Two Methods + Comparison

## Part 1: Data Loading & EDA Summary

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Setup paths
ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
DATA_PATH = os.path.join(ROOT, 'data', 'processed', 'model_data_customers.csv')
OUT_DIR = os.path.join(ROOT, 'outputs', 'data', 'analysis')
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Loading data from: {DATA_PATH}")
df = pd.read_csv(DATA_PATH)
print(f"✓ Data loaded: {df.shape[0]} customers, {df.shape[1]} features")

In [ ]:
# EDA Summary
print("\n" + "="*60)
print("EXPLORATORY DATA ANALYSIS SUMMARY")
print("="*60)

print("\n📊 DATASET OVERVIEW:")
print(df.info())
print("\n📈 DESCRIPTIVE STATISTICS:")
print(df.describe())

print("\n🎯 TARGET VARIABLE DISTRIBUTION:")
print(df['Label'].value_counts())
print(f"\nClass Distribution:")
print(f"  - Non-Champions (0): {(df['Label']==0).sum()} ({(df['Label']==0).mean()*100:.1f}%)")
print(f"  - Champions (1): {(df['Label']==1).sum()} ({(df['Label']==1).mean()*100:.1f}%)")

In [ ]:
# Visualization 1: Target Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar plot
df['Label'].value_counts().plot(kind='bar', ax=axes[0], color=['#FF6B6B', '#4ECDC4'])
axes[0].set_title('Customer Classification Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Customer Type')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Non-Champions', 'Champions'], rotation=0)

# Pie chart
sizes = df['Label'].value_counts().values
labels = ['Non-Champions', 'Champions']
colors = ['#FF6B6B', '#4ECDC4']
axes[1].pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Class Distribution', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'eda_target_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: eda_target_distribution.png")

In [ ]:
# Visualization 2: Feature Distribution by Class
feature_cols = ['Recency', 'Frequency', 'Monetary', 'AvgOrderValue', 'AvgItemsPerOrder', 'DistinctProducts']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for idx, feat in enumerate(feature_cols):
    df[df['Label']==0][feat].hist(bins=30, alpha=0.6, label='Non-Champions', ax=axes[idx], color='#FF6B6B')
    df[df['Label']==1][feat].hist(bins=30, alpha=0.6, label='Champions', ax=axes[idx], color='#4ECDC4')
    axes[idx].set_title(f'Distribution: {feat}', fontweight='bold')
    axes[idx].set_xlabel('Value')
    axes[idx].set_ylabel('Frequency')
    axes[idx].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'eda_feature_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: eda_feature_distributions.png")

## Part 2: Method 1 - Correlation-Based Feature Selection

In [ ]:
print("\n" + "="*60)
print("METHOD 1: CORRELATION-BASED FEATURE SELECTION")
print("="*60)

# Prepare data
feature_cols = ['Recency', 'Frequency', 'Monetary', 'AvgOrderValue', 'AvgItemsPerOrder', 'DistinctProducts']
X = df[feature_cols].fillna(0)
y = df['Label']

# Standardize features (important for correlation-based selection)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\n📊 Analyzing {len(feature_cols)} features...")
print(f"   Features: {', '.join(feature_cols)}")

In [ ]:
# Correlation Analysis with Target
correlations = []
for col in feature_cols:
    corr = np.corrcoef(df[col], y)[0, 1]
    correlations.append(corr)

corr_df = pd.DataFrame({
    'Feature': feature_cols,
    'Correlation': correlations,
    'Abs_Correlation': np.abs(correlations)
}).sort_values('Abs_Correlation', ascending=False)

print("\n🔗 CORRELATION WITH TARGET (Champions vs Non-Champions):")
print(corr_df.to_string(index=False))

In [ ]:
# SelectKBest with f_classif scoring
selector_kbest = SelectKBest(score_func=f_classif, k='all')
selector_kbest.fit(X_scaled, y)

kbest_scores = pd.DataFrame({
    'Feature': feature_cols,
    'F-Score': selector_kbest.scores_,
    'P-Value': selector_kbest.pvalues_
}).sort_values('F-Score', ascending=False)

print("\n📊 SELECTKBEST - F-STATISTIC SCORES:")
print(kbest_scores.to_string(index=False))

# Determine number of features to select (e.g., top 3-4)
top_k = 3
selected_features_method1 = kbest_scores['Feature'].head(top_k).tolist()
print(f"\n✓ Selected Top {top_k} Features (Correlation-Based):")
for i, feat in enumerate(selected_features_method1, 1):
    score = kbest_scores[kbest_scores['Feature']==feat]['F-Score'].values[0]
    print(f"  {i}. {feat} (F-Score: {score:.2f})")

In [ ]:
# Visualization 3: Correlation-Based Selection
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F-Scores
colors_method1 = ['#2ECC71' if feat in selected_features_method1 else '#95A5A6' for feat in kbest_scores['Feature']]
axes[0].barh(kbest_scores['Feature'], kbest_scores['F-Score'], color=colors_method1)
axes[0].set_xlabel('F-Score', fontweight='bold')
axes[0].set_title('Method 1: F-Statistics Scores (SelectKBest)', fontweight='bold', fontsize=12)
axes[0].invert_yaxis()

# Correlation
colors_corr = ['#E74C3C' if corr > 0 else '#3498DB' for corr in corr_df['Correlation']]
axes[1].barh(corr_df['Feature'], corr_df['Correlation'], color=colors_corr)
axes[1].set_xlabel('Correlation Coefficient', fontweight='bold')
axes[1].set_title('Feature Correlation with Target', fontweight='bold', fontsize=12)
axes[1].invert_yaxis()
axes[1].axvline(x=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'method1_correlation_based.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: method1_correlation_based.png")

## Part 3: Method 2 - Backward Elimination

In [ ]:
print("\n" + "="*60)
print("METHOD 2: BACKWARD ELIMINATION")
print("="*60)

print("\n📋 BACKWARD ELIMINATION PROCESS:")
print("   Algorithm: Iteratively remove features with lowest importance")
print("   Estimator: Logistic Regression (for interpretability)")
print("   Target: Stop at k=3 features\n")

# Use Recursive Feature Elimination with Logistic Regression
estimator = LogisticRegression(random_state=42, max_iter=1000)
rfe = RFE(estimator=estimator, n_features_to_select=3, step=1)
rfe.fit(X_scaled, y)

# Get feature rankings
rfe_df = pd.DataFrame({
    'Feature': feature_cols,
    'Ranking': rfe.ranking_,
    'Selected': rfe.support_
}).sort_values('Ranking')

print("\n🔢 RFE RANKINGS (Lower = Better):")
print(rfe_df.to_string(index=False))

selected_features_method2 = rfe_df[rfe_df['Selected']]['Feature'].tolist()
print(f"\n✓ Selected {len(selected_features_method2)} Features (Backward Elimination):")
for i, feat in enumerate(selected_features_method2, 1):
    rank = rfe_df[rfe_df['Feature']==feat]['Ranking'].values[0]
    print(f"  {i}. {feat} (Ranking: {rank})")

In [ ]:
# Get feature coefficients from fitted Logistic Regression (refit to get coef_)
estimator_coef = LogisticRegression(random_state=42, max_iter=1000)
estimator_coef.fit(X_scaled, y)
coefficients = np.abs(estimator_coef.coef_[0])
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': coefficients,
    'Importance': coefficients / coefficients.sum()  # Normalized
}).sort_values('Coefficient', ascending=False)

print("📊 LOGISTIC REGRESSION COEFFICIENTS (Feature Importance):")
print(coef_df.to_string(index=False))

In [ ]:
# Visualization 4: Backward Elimination Process
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RFE Rankings
colors_rfe = ['#2ECC71' if feat in selected_features_method2 else '#95A5A6' for feat in rfe_df['Feature']]
axes[0].barh(rfe_df['Feature'], 7 - rfe_df['Ranking'], color=colors_rfe)
axes[0].set_xlabel('Feature Importance (Inverse Ranking)', fontweight='bold')
axes[0].set_title('Method 2: Backward Elimination - RFE Ranking', fontweight='bold', fontsize=12)
axes[0].invert_yaxis()

# Logistic Regression Coefficients
colors_coef = ['#2ECC71' if feat in selected_features_method2 else '#95A5A6' for feat in coef_df['Feature']]
axes[1].barh(coef_df['Feature'], coef_df['Coefficient'], color=colors_coef)
axes[1].set_xlabel('|Coefficient| Value', fontweight='bold')
axes[1].set_title('Logistic Regression Coefficients', fontweight='bold', fontsize=12)
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'method2_backward_elimination.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: method2_backward_elimination.png")

## Part 4: Comparison of Methods

In [ ]:
print("\n" + "="*60)
print("METHOD COMPARISON")
print("="*60)

# Create comparison DataFrame
comparison_data = []
for feat in feature_cols:
    m1_score = kbest_scores[kbest_scores['Feature']==feat]['F-Score'].values[0]
    m1_selected = 'Yes' if feat in selected_features_method1 else 'No'
    
    m2_rank = rfe_df[rfe_df['Feature']==feat]['Ranking'].values[0]
    m2_selected = 'Yes' if feat in selected_features_method2 else 'No'
    
    m2_coef = coef_df[coef_df['Feature']==feat]['Coefficient'].values[0]
    
    comparison_data.append({
        'Feature': feat,
        'Method 1: F-Score': round(m1_score, 2),
        'M1 Selected': m1_selected,
        'Method 2: RFE Rank': int(m2_rank),
        'Method 2: Coef': round(m2_coef, 4),
        'M2 Selected': m2_selected
    })

comparison_table = pd.DataFrame(comparison_data).sort_values('Feature')

print("\n📊 DETAILED COMPARISON TABLE:")
print(comparison_table.to_string(index=False))

In [ ]:
# Summary Statistics
print("\n" + "-"*60)
print("SELECTED FEATURES COMPARISON:")
print("-"*60)

print(f"\n✓ METHOD 1 - Correlation-Based (SelectKBest):")
print(f"   Selected: {selected_features_method1}")

print(f"\n✓ METHOD 2 - Backward Elimination (RFE):")
print(f"   Selected: {selected_features_method2}")

# Find intersection and differences
common_features = set(selected_features_method1) & set(selected_features_method2)
unique_m1 = set(selected_features_method1) - set(selected_features_method2)
unique_m2 = set(selected_features_method2) - set(selected_features_method1)

print(f"\n🔍 FEATURE OVERLAP:")
print(f"   Common features (both methods): {list(common_features)}")
print(f"   Unique to Method 1: {list(unique_m1)}")
print(f"   Unique to Method 2: {list(unique_m2)}")
print(f"   Agreement rate: {len(common_features)}/3 = {len(common_features)/3*100:.1f}%")

In [ ]:
# Visualization 5: Side-by-Side Comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Selected Features Venn Diagram (alternative: bar chart)
all_features = set(feature_cols)
methods_comparison = pd.DataFrame({
    'Feature': feature_cols,
    'Method 1\n(SelectKBest)': [1 if f in selected_features_method1 else 0 for f in feature_cols],
    'Method 2\n(RFE)': [1 if f in selected_features_method2 else 0 for f in feature_cols]
})

x = np.arange(len(feature_cols))
width = 0.35
axes[0, 0].bar(x - width/2, methods_comparison['Method 1\n(SelectKBest)'], width, label='SelectKBest', color='#3498DB')
axes[0, 0].bar(x + width/2, methods_comparison['Method 2\n(RFE)'], width, label='RFE', color='#E74C3C')
axes[0, 0].set_ylabel('Selected (1) / Not Selected (0)', fontweight='bold')
axes[0, 0].set_title('Selected Features Comparison', fontweight='bold', fontsize=12)
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(feature_cols, rotation=45, ha='right')
axes[0, 0].legend()
axes[0, 0].set_ylim([0, 1.2])

# 2. Feature Scores Normalization (for comparison)
norm_m1 = (kbest_scores.set_index('Feature')['F-Score'] / kbest_scores['F-Score'].max()).sort_index()
norm_m2 = (coef_df.set_index('Feature')['Coefficient'] / coef_df['Coefficient'].max()).sort_index()

x = np.arange(len(feature_cols))
axes[0, 1].bar(x - width/2, norm_m1, width, label='SelectKBest (Normalized)', color='#3498DB')
axes[0, 1].bar(x + width/2, norm_m2, width, label='RFE (Normalized)', color='#E74C3C')
axes[0, 1].set_ylabel('Normalized Score', fontweight='bold')
axes[0, 1].set_title('Feature Importance Scores (Normalized)', fontweight='bold', fontsize=12)
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(feature_cols, rotation=45, ha='right')
axes[0, 1].legend()

# 3. Method Characteristics
method_chars = [
    ['Approach', 'Filter-based\nStatistical', 'Wrapper-based\nIterative'],
    ['Speed', 'Fast', 'Slower'],
    ['Interpretability', 'High', 'Medium'],
    ['Computational Cost', 'Low', 'High'],
    ['Best for', 'Large datasets', 'Small datasets']
]

axes[1, 0].axis('tight')
axes[1, 0].axis('off')
table = axes[1, 0].table(
    cellText=method_chars,
    colLabels=['Characteristic', 'Method 1: SelectKBest', 'Method 2: RFE'],
    cellLoc='center',
    loc='center',
    bbox=[0, 0, 1, 1]
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 2)
axes[1, 0].set_title('Method Characteristics', fontweight='bold', fontsize=12, pad=20)

# 4. Recommendation
axes[1, 1].axis('off')
recommendation_text = f"""
🏆 RECOMMENDATION FOR THIS PROJECT:

Selected Method: METHOD 1 (SelectKBest)

Reasons:
✓ Fast and efficient for this dataset
✓ Clear statistical significance (F-scores)
✓ Top 3 Features: {', '.join(selected_features_method1)}
✓ Easy to interpret and explain

Alternative: METHOD 2 (RFE)
✓ Provides stable feature rankings
✓ Good for model validation
✓ Top 3 Features: {', '.join(selected_features_method2)}

Consensus Features (both methods):
→ {list(common_features)}
"""

axes[1, 1].text(0.05, 0.95, recommendation_text, 
                 transform=axes[1, 1].transAxes,
                 fontsize=10,
                 verticalalignment='top',
                 family='monospace',
                 bbox=dict(boxstyle='round', facecolor='#F0F0F0', alpha=0.8))

plt.suptitle('Feature Selection Methods - Comprehensive Comparison', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'feature_selection_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: feature_selection_comparison.png")

In [ ]:
# Save comparison table to CSV
comparison_table.to_csv(os.path.join(OUT_DIR, 'feature_selection_comparison_table.csv'), index=False)
print("✓ Saved: feature_selection_comparison_table.csv")

## Part 5: Model Performance with Selected Features

In [ ]:
print("\n" + "="*60)
print("MODEL PERFORMANCE WITH SELECTED FEATURES")
print("="*60)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

results = []

# Test different feature sets
feature_sets = {
    'All Features (6)': feature_cols,
    'Method 1: SelectKBest (3)': selected_features_method1,
    'Method 2: RFE (3)': selected_features_method2,
}

for set_name, features in feature_sets.items():
    # Get feature indices
    feat_indices = [feature_cols.index(f) for f in features]
    X_train_subset = X_train[:, feat_indices]
    X_test_subset = X_test[:, feat_indices]
    
    # Train RandomForest
    clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    clf.fit(X_train_subset, y_train)
    
    # Predict
    y_pred = clf.predict(X_test_subset)
    y_proba = clf.predict_proba(X_test_subset)[:, 1]
    
    # Evaluate
    roc_auc = roc_auc_score(y_test, y_proba)
    
    results.append({
        'Feature Set': set_name,
        'Num Features': len(features),
        'ROC-AUC': round(roc_auc, 4),
        'Features': ', '.join(features)
    })
    
    print(f"\n{set_name}:")
    print(f"  ROC-AUC: {roc_auc:.4f}")
    print(f"  Features: {', '.join(features)}")

results_df = pd.DataFrame(results)
print("\n" + "="*60)
print("PERFORMANCE SUMMARY:")
print(results_df.to_string(index=False))

In [ ]:
# Visualization 6: Model Performance Comparison
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#95A5A6', '#3498DB', '#E74C3C']
bars = ax.bar(results_df['Feature Set'], results_df['ROC-AUC'], color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}',
            ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_ylabel('ROC-AUC Score', fontweight='bold', fontsize=12)
ax.set_title('Model Performance Comparison\n(RandomForest Classifier)', fontweight='bold', fontsize=13)
ax.set_ylim([0, 1.0])
ax.axhline(y=0.9, color='green', linestyle='--', alpha=0.5, label='Excellent (0.9+)')
ax.legend()

plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'model_performance_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: model_performance_comparison.png")

## Part 6: Final Recommendation & Conclusions

In [ ]:
print("\n" + "="*60)
print("FINAL CONCLUSIONS & RECOMMENDATIONS")
print("="*60)

print(f"""
📋 EXECUTIVE SUMMARY:

1️⃣ METHOD 1 - CORRELATION-BASED (SelectKBest)
   ✓ Algorithm: Statistical F-test (ANOVA)
   ✓ Approach: Filter-based, univariate
   ✓ Selected Features: {', '.join(selected_features_method1)}
   ✓ Model ROC-AUC: {results_df[results_df['Feature Set'].str.contains('Method 1')]['ROC-AUC'].values[0]:.4f}
   
   Pros:
   - Fast computational speed
   - Easy to interpret (F-scores explain variance)
   - Statistically rigorous
   - No multicollinearity issues
   
   Cons:
   - Ignores feature interactions
   - Univariate only (no joint effects)


2️⃣ METHOD 2 - BACKWARD ELIMINATION (RFE)
   ✓ Algorithm: Recursive Feature Elimination
   ✓ Approach: Wrapper-based, iterative removal
   ✓ Selected Features: {', '.join(selected_features_method2)}
   ✓ Model ROC-AUC: {results_df[results_df['Feature Set'].str.contains('Method 2')]['ROC-AUC'].values[0]:.4f}
   
   Pros:
   - Considers feature interactions
   - Wrapper-based (uses actual model)
   - More stable for ML tasks
   - Better for non-linear problems
   
   Cons:
   - Computationally expensive
   - Risk of overfitting on training data
   - Less interpretable than filter methods


🏆 RECOMMENDED METHOD:
   → METHOD 1 (SelectKBest - Correlation-Based)
   
   Reasons:
   ✓ Excellent ROC-AUC performance ({results_df[results_df['Feature Set'].str.contains('Method 1')]['ROC-AUC'].values[0]:.4f})
   ✓ Only 3 features needed (vs 6 all features)
   ✓ Significant dimension reduction (50%)
   ✓ Highly interpretable for business stakeholders
   ✓ Faster inference in production
   ✓ Easier to explain model decisions


📊 CONSENSUS FEATURES (both methods agree):
   → {list(common_features)}
   
   These features are CRITICAL for champion prediction!


💡 BUSINESS INSIGHTS:
   • DistinctProducts (variety): Customers buying diverse products are likely champions
   • Frequency: More transactions = higher loyalty indicator  
   • Recency: Recent activity = engaged customer (positive indicator)


📈 NEXT STEPS:
   1. Use top 3 features from Method 1 in final model
   2. Implement cross-validation for robustness
   3. Test other algorithms (XGBoost, SVM)
   4. Deploy model with selected features
   5. Monitor feature importance over time
""")

In [ ]:
# Save all results to summary CSV
summary_data = {
    'Analysis': ['EDA Complete', 'Method 1 (SelectKBest)', 'Method 2 (RFE)', 'Comparison', 'Recommendation'],
    'Status': ['✓ Done', '✓ Done', '✓ Done', '✓ Done', '✓ Method 1 Recommended'],
    'Features Selected': ['-', str(selected_features_method1), str(selected_features_method2), '-', str(selected_features_method1)],
    'Model ROC-AUC': ['-', f"{results_df[results_df['Feature Set'].str.contains('Method 1')]['ROC-AUC'].values[0]:.4f}", 
                      f"{results_df[results_df['Feature Set'].str.contains('Method 2')]['ROC-AUC'].values[0]:.4f}", '-', 
                      f"{results_df[results_df['Feature Set'].str.contains('Method 1')]['ROC-AUC'].values[0]:.4f}"]
}

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(os.path.join(OUT_DIR, 'feature_selection_summary.csv'), index=False)
print("✓ Saved: feature_selection_summary.csv")

print("\n" + "="*60)
print("✅ NOTEBOOK EXECUTION COMPLETE")
print("="*60)
print(f"\n📁 All outputs saved to: {OUT_DIR}")
print("\nGenerated Files:")
print("  - eda_target_distribution.png")
print("  - eda_feature_distributions.png")
print("  - method1_correlation_based.png")
print("  - method2_backward_elimination.png")
print("  - feature_selection_comparison.png")
print("  - model_performance_comparison.png")
print("  - feature_selection_comparison_table.csv")
print("  - feature_selection_summary.csv")